# Computational Tools for Climate Science - Project 2026
---
**Group 2 - Chaac**

Authors: *Abiyo Gladys, Alex Blackmer, Caroline Roper, Han Nguyen, Negli Gallardo, Sheila Aguilar Palpa*

Date: *July 13-24th*

---

This project explores the relationship between precipitation and crop production in Southeast Asia. Specifically, we tie the Standardized Precipitation Index (SPI) derived from Climate Hazards Group InfraRed Precipitation with Station data (CHIRPS) with cereal production yield anomalies sourced from the Food and Agriculture Organization (FAO). This study considers two domains in Southeast Asia; the mainland (Monsoon Climate Region - MCR) and maritime (Equatorial Climate Region - ECR), focusing on the countries of Vietnam and Indonesia respectively.

In [1]:
# Imports
import os
# Scientific data packages
import numpy as np
import xarray as xr
import pandas as pd
import datetime
# Plotting libraries
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
# Climate index package
import xclim
from scipy.stats import pearsonr
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [16]:
import sys
sys.path.append('./Code')

from util import config, io, annual_spi_metrics_util, correlation_util
from util.io import load_chirps, load_country_mask, load_wdi_cereal

In [3]:
precip_ds = load_chirps()
precip_ds

<xarray.Dataset> Size: 1GB
Dimensions:    (time: 546, latitude: 800, longitude: 800)
Coordinates:
  * time       (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2026-06-01
  * latitude   (latitude) float32 3kB -19.98 -19.92 -19.88 ... 19.88 19.92 19.97
  * longitude  (longitude) float32 3kB 100.0 100.1 100.1 ... 139.9 139.9 140.0
Data variables:
    precip     (time, latitude, longitude) float32 1GB ...
Attributes: (12/14)
    Conventions:       CF-1.6
    title:             CHIRPS Version 3.0
    history:           created by Climate Hazards Center
    version:           Version 3.0
    data_created:      2026-07-14
    creator_name:      Pete Peterson
    ...                ...
    documentation:     http://pubs.usgs.gov/ds/832/
    reference:         Funk, C.C., Peterson, P.J., Landsfeld, M.F., Pedreros,...
    acknowledgements:  The Climate Hazards Center InfraRed Precipitation with...
    ftp_url:           ftp://chg-ftpout.geog.ucsb.edu/pub/chg/products/CHIRPS...
    website:           http://chg.geog.ucsb.edu/data/chirps/index.html
    faq:               http://chg-wiki.geog.ucsb.edu/wiki/CHIRPS_FAQ

### Country Mask

In [4]:
country_mask = load_country_mask()
country_mask = country_mask.rename({"lat": "latitude", "lon": "longitude"})

country_mask = country_mask["country_mask"].interp(
    latitude=precip_ds.latitude,
    longitude=precip_ds.longitude,
    method="nearest"
)

### Cereal Production

In [5]:
cereal_df = load_fao_crops()

---
## Subset Precipitation Data By Country

In [6]:
# Mask each contry area and drop the NaNs
precip_ind = precip_ds.precip.where(country_mask == 1, drop=True)
precip_viet = precip_ds.precip.where(country_mask == 2, drop=True)

---
# Analysis

## SPI Calculation
Standardized precipitation index is calculated individually for Vietnam and Indonesia

In [7]:
precip_viet_1d = precip_viet.mean(dim=["latitude", "longitude"])
spi_1m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=1)
spi_6m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=6)
spi_12m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=12)

/Users/carolineroper/opt/anaconda3/envs/drought_severity/lib/python3.13/site-packages/xclim/core/cfchecks.py:80: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/Users/carolineroper/opt/anaconda3/envs/drought_severity/lib/python3.13/site-packages/xclim/core/cfchecks.py:80: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/Users/carolineroper/opt/anaconda3/envs/drought_severity/lib/python3.13/site-packages/xclim/core/cfchecks.py:80: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])


In [8]:
precip_ind_1d = precip_ind.mean(dim=["latitude", "longitude"])
spi_1m_ind = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_ind_1d, freq='MS', window=1)
spi_6m_ind = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_ind_1d, freq='MS', window=6)
spi_12m_ind = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_ind_1d, freq='MS', window=12)

/Users/carolineroper/opt/anaconda3/envs/drought_severity/lib/python3.13/site-packages/xclim/core/cfchecks.py:80: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/Users/carolineroper/opt/anaconda3/envs/drought_severity/lib/python3.13/site-packages/xclim/core/cfchecks.py:80: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/Users/carolineroper/opt/anaconda3/envs/drought_severity/lib/python3.13/site-packages/xclim/core/cfchecks.py:80: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])


In [9]:
per_year_months_at_severity_level_viet = annual_spi_metrics_util.monthly_severity_per_year(spi_1m_viet.spi)

In [10]:
per_year_months_at_severity_level_indonesia = annual_spi_metrics_util.monthly_severity_per_year(spi_1m_ind.spi)

In [11]:
spi_yearly_viet = annual_spi_metrics_util.categorize_spi(spi_12m_viet.spi)
spi_yearly_indonesia = annual_spi_metrics_util.categorize_spi(spi_12m_ind.spi)

In [12]:
annual_spi_viet = per_year_months_at_severity_level_viet.join(spi_yearly_viet)
#annual_spi_viet.to_csv(path_gd + 'annual_spi_vietnam.csv')

In [13]:
annual_spi_indonesia = per_year_months_at_severity_level_indonesia.join(spi_yearly_indonesia)
#annual_spi_indonesia.to_csv(path_gd + 'annual_spi_indonesia.csv')

## Cereal Anomalies

In [17]:
#Creating a dataset
#cereal_land = pd.read_csv(path + "data_cereal_land.csv", skiprows=4, header=0)
cereal_land = load_wdi_cereal()

In [18]:
#Selectiong countries and columns of interest
countries = ['Indonesia', 'Viet Nam']
columns_of_interest = ['Country Name', 'Country Code'] + [str(year) for year in range(1981, 2024)]

In [19]:
cereal_land.head()

,Country Name,Country Code,Year,Production
0,Indonesia,IDN,1981,37283478.0
1,Viet Nam,VNM,1981,12844800.0
2,Indonesia,IDN,1982,36818521.0
3,Viet Nam,VNM,1982,14828300.0
4,Indonesia,IDN,1983,40389883.0


In [20]:
cereal_land_countries = cereal_land[cereal_land['Country Name'].isin(countries)]

In [21]:
existing_columns = [col for col in columns_of_interest if col in cereal_land_countries.columns]
cereal_land_filtered = cereal_land_countries[existing_columns]

In [22]:
cereal_land_long_VI = cereal_land_filtered.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='Year',
    value_name='Production'
)

In [23]:
#Calculating the mean production of cereal
mean_production_country = cereal_land_long_VI.groupby('Country Name')['Production'].mean().reset_index()
mean_production_country = mean_production_country.rename(columns={'Production': 'Production_Mean'})

In [24]:
#Merging the mean to the dataset
cereal_land_long_VI = cereal_land_long_VI.merge(mean_production_country, on='Country Name')

In [25]:
#Calculating anomaly
cereal_land_long_VI['Anomaly'] = cereal_land_long_VI['Production'] - cereal_land_long_VI['Production_Mean']

Normalizing cereal production anomaly.

In [26]:
#Calculating standar deviation for each country (original time series)
std_by_country = cereal_land_long_VI.groupby('Country Name')['Production'].std().reset_index()
std_by_country = std_by_country.rename(columns={'Production': 'Production_Std'})

In [27]:
#Merge new variable
cereal_land_long_VI = cereal_land_long_VI.merge(std_by_country, on='Country Name')

In [28]:
#Normalizing anomaly (Z-score)
cereal_land_long_VI['Anomaly_Normalized'] = cereal_land_long_VI['Anomaly'] / cereal_land_long_VI['Production_Std']

In [31]:
#Save dataset as CSV file
path_drive = "data/"
cereal_land_long_VI.to_csv(path_drive + 'normalized_anomaly_cereal_data.csv', index=False)

## SPI & Cereal Correlation
TODO: Individually correlate each SPI series (1m, 6m, 12m) with cereal anomalies to assess which has strongest relationship. Do for each domain.

## Drought Severity Analysis
TODO: Subset each SPI series by drought severity category to then assess relationship between drought severity and crop production. Can create scatter plots and time series. Do for each domain.

In [ ]:
# crops_ind = cereal_land_long_VI.loc[cereal_land_long_VI['Country Name'] == 'Indonesia', :].set_index('Year') # this is our data that is NOT de-trended

In [44]:
detrended = pd.read_csv('data/' + 'detrended_normalized_anomaly_cereal_data.csv')

In [45]:
detrended.head()

,Country Name,Country Code,Year,Production,detrended,detrended_Mean,Anomaly,detrended_Std,Anomaly_Normalized
0,Indonesia,IDN,1981,37283478.0,-1.433609e+06,4.851541e-09,-1.433609e+06,2.158616e+06,-0.664133
1,Viet Nam,VNM,1981,12844800.0,1.973415e+06,1.433804e-08,1.973415e+06,1.203843e+06,1.639262
2,Indonesia,IDN,1982,36818521.0,-3.127823e+06,4.851541e-09,-3.127823e+06,2.158616e+06,-1.448995
3,Viet Nam,VNM,1982,14828300.0,2.774720e+06,1.433804e-08,2.774720e+06,1.203843e+06,2.304885
4,Indonesia,IDN,1983,40389883.0,-7.857188e+05,4.851541e-09,-7.857188e+05,2.158616e+06,-0.363992


In [46]:
#detrended = add_anomaly(detrended, 'detrended')

In [47]:
crops_and_spi_ind = correlation_util.prep_corr(detrended, annual_spi_indonesia, 'Indonesia')

In [48]:
crops_and_spi_viet = correlation_util.prep_corr(detrended, annual_spi_viet, 'Viet Nam')

In [49]:
crops_and_spi_ind.groupby('category')['Anomaly_Normalized'].agg(['mean', 'median'])

/var/folders/p5/zvnrh_7s1fjcmppf664d_pgh0000gn/T/ipykernel_40365/738625448.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  crops_and_spi_ind.groupby('category')['Anomaly_Normalized'].agg(['mean', 'median'])


,mean,median
category,,
Extreme Drought,-0.109931,-0.109931
Severe Drought,-0.855763,-1.448995
Moderate Drought,-0.271107,-0.397829
Mild Drought,0.138629,0.040192
No Drought,0.107691,0.199426


In [50]:
#independence assumption may be violated here

In [51]:
# Define the model
model = ols('Anomaly ~ C(category)', data=crops_and_spi_ind).fit()

# Perform ANOVA
anova_table = sm.stats.anova_lm(model, typ=1)
print(anova_table)

               df        sum_sq       mean_sq         F    PR(>F)
C(category)   4.0  1.426930e+13  3.567325e+12  0.747146  0.566035
Residual     38.0  1.814349e+14  4.774603e+12       NaN       NaN


In [52]:
crops_and_spi_ind[['spi', 'Anomaly_Normalized']].corr()

,spi,Anomaly_Normalized
spi,1.000000,0.286378
Anomaly_Normalized,0.286378,1.000000


In [60]:
correlation_coefficient, p_value = correlation_util.calculate_pearson_correlation(
    crops_and_spi_ind.spi.values, crops_and_spi_ind.Anomaly_Normalized.values
)

In [61]:
correlation_coefficient

np.float64(0.28637846002297873)

In [62]:
correlation_coefficient, p_value = correlation_util.calculate_pearson_correlation(
    crops_and_spi_viet.spi.values, crops_and_spi_viet.Anomaly_Normalized.values
)

In [63]:
correlation_coefficient

np.float64(0.0339202827047418)